# Time Travel
This notebook is my exploration of the [Time Travel](https://langchain-ai.github.io/langgraph/tutorials/get-started/6-time-travel/) tutorial by Langgraph. It follows the previous notebook that [added a customised state](./chatbot_with_customised_state.ipynb) to a chatbot. In this notebook, we will understand how to fork the execution of a graph to rewind and either fix mistakes or try a different strategy.

## Required packages
* `langchain[openai]`
* `langchain-tavily`
* `langgraph`
* `langgraph-checkpoint-sqlite`
* `langsmith`

In [1]:
import json
import sqlite3
import uuid
from typing import Annotated, Union
from typing_extensions import TypedDict

from IPython.display import display, Markdown
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.sqlite import SqliteSaver 
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command

In [2]:
# Load OpenAI and Tavily API keys
# The `secrets.env` file is expected to have the following lines:
# OPENAI_API_KEY="sk-proj-xxxxx"
# TAVILY_API_KEY="tvly-dev-xxxxx"
from dotenv import load_dotenv
load_dotenv("../../secrets.env")

True

## Graph with memory
Let us build a graph with memory.

In [3]:
search_tool = TavilySearch(max_results=2)
tools = [search_tool]

In [4]:
llm = init_chat_model("openai:gpt-5-mini")

class State(TypedDict):
    messages: Annotated[list, add_messages]

# Tell the LLM the tools it can call
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])
    return {"messages": [message]}

In [5]:
graph_builder = StateGraph(State)

graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition
)

graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

In [6]:
# Note: check_same_thread=False is OK as the implementation uses a lock
# to ensure thread safety.
conn = sqlite3.connect("chatbot_time_travel.db", check_same_thread=False)
memory = SqliteSaver(conn)

graph = graph_builder.compile(checkpointer=memory)

## Prompt the chatbot
Let us now have a multi-turn conversation with the chatbot.

In [7]:
config = {"configurable": {"thread_id": uuid.uuid4()}}

In [8]:
def display_text_response(text: str) -> None:
    """Display a Markdown-formatted assistant response."""
    display(Markdown("**Assistant:**\n" + text))

def display_search_tool_input(tool_input: dict) -> str:
    """Format search tool input for display.
    
    Args:
        tool_input (dict): Dictionary containing search tool input details.
        
    Returns:
        str: Formatted markdown string for search tool input.
    """
    lines = []
    
    query = tool_input.get("query", "")
    if query:
        lines.append(f"- **Query:** `{query}`")
    
    topic = tool_input.get("topic", "")
    if topic:
        lines.append(f"- **Topic:** `{topic}`")
    
    search_depth = tool_input.get("search_depth", "")
    if search_depth:
        lines.append(f"- **Search Depth:** `{search_depth}`")
    
    return "\n".join(lines)

def display_human_assistance_tool_input(tool_input: dict) -> str:
    """Format human assistance tool input for display.
    
    Args:
        tool_input (dict): Dictionary containing human assistance tool input details.
        
    Returns:
        str: Formatted markdown string for human assistance tool input.
    """
    lines = []
    
    name = tool_input.get("name", "")
    if name:
        lines.append(f"- **Name:** `{name}`")
    
    birthday = tool_input.get("birthday", "")
    if birthday:
        lines.append(f"- **Birthday:** `{birthday}`")
    
    return "\n".join(lines)

def display_tool_use(tool_info: dict) -> None:
    """Display information about a tool usage in Markdown format.
    
    Args:
        tool_info (dict): Dictionary containing tool usage details.
    """
    tool_name = tool_info.get("name", "")
    tool_input = tool_info.get("input", {})
    
    # Determine which type of tool input we have and format accordingly
    if "query" in tool_input or "topic" in tool_input or "search_depth" in tool_input:
        input_display = display_search_tool_input(tool_input)
    elif "name" in tool_input or "birthday" in tool_input:
        input_display = display_human_assistance_tool_input(tool_input)
    else:
        input_display = ""
    
    markdown_content = f"**Tool Use:** `{tool_name}`\n\n{input_display}\n" if input_display else f"**Tool Use:** `{tool_name}`\n"
    display(Markdown(markdown_content))

def display_search_results(results: list) -> None:
    """Display a list of search results in Markdown format.
    
    Args:
        results (list): List of dictionaries containing search result details.
    """
    for result in results:
        url = result.get("url", "")
        title = result.get("title", "")
        score = result.get("score", "")
        published_date = result.get("published_date", "")
        
        lines = ["**Search Result:**"]
        
        if title and url:
            lines.append(f"- **Title:** [{title}]({url})")
        elif title:
            lines.append(f"- **Title:** {title}")
        
        if score:
            lines.append(f"- **Score:** `{score}`")
        
        if published_date:
            lines.append(f"- **Published Date:** `{published_date}`")
        
        display(Markdown("\n".join(lines) + "\n"))

def handle_latest_message(latest_message: object) -> None:
    """Handle and display the latest message from the graph event.
    
    Args:
        latest_message (object): The latest message, can be a list or string.
    """
    if isinstance(latest_message, list):
        for v in latest_message:
            if v["type"] == "text":
                display_text_response(v["text"])
            elif v["type"] == "tool_use":
                display_tool_use(v)
            else:
                raise ValueError("Message type is neither 'text' nor 'tool_use'")
    elif isinstance(latest_message, str):
        try:
            results = json.loads(latest_message)["results"]
            display_search_results(results)
        except json.JSONDecodeError:
            display_text_response(latest_message)
    else:
        raise ValueError("Latest message is neither of 'str' or 'list'")

def stream_graph_updates(user_input: Union[str, Command], config: dict) -> None:
    """Stream updates from the graph for a given user input and display responses.
    
    Args:
        user_input (Union[str, Command]): The user's input message or command.
    """
    if isinstance(user_input, str):
        stream_input = {"messages": [{"role": "user", "content": user_input}]}
    elif isinstance(user_input, Union[Command, None]):
        stream_input = user_input
    else:
        raise ValueError("Invalid user input type")

    for event in graph.stream(
        stream_input,
        config=config
    ):
        for value in event.values():
            latest_message = value["messages"][-1].content
            handle_latest_message(latest_message)

In [9]:
while True:
    user_input = input("User: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    display(Markdown(f"**Question**:\n{user_input}\n"))
    try:
        stream_graph_updates(user_input, config)
    except TypeError:
        display(Markdown(f"Waiting for human assistance..."))
        break

**Question**:
Tell me about Japan in 3 sentences.


**Assistant:**
Japan is an island nation in East Asia made up of four main islands—Honshu, Hokkaido, Kyushu and Shikoku—and thousands of smaller ones, with Tokyo as its capital; it is mountainous and lies on the Pacific Ring of Fire, so earthquakes and volcanoes are common. It has a highly developed, export-oriented economy known for automobiles, electronics, robotics and precision manufacturing, along with advanced infrastructure and a high standard of living. Japan's culture blends long-standing traditions—Shinto and Buddhist practices, tea ceremony, calligraphy and seasonal festivals—with vibrant modern pop culture like anime and manga; politically it is a constitutional monarchy with an emperor and a parliamentary government.

**Question**:
Which are the 5 largest cities in Japan?


**Assistant:**
If you mean by population of the municipality (city proper), the five largest cities in Japan are:

1. Yokohama — ~3.7 million  
2. Osaka — ~2.7 million  
3. Nagoya — ~2.3 million  
4. Sapporo — ~1.9 million  
5. Fukuoka — ~1.6 million

Note: Tokyo's 23 special wards together have about 9.7 million people (and Tokyo Metropolis as a whole ~14 million), so Tokyo is the largest urban area if you group wards or use metropolitan-area figures. Do you want the largest by metropolitan area instead or exact recent census numbers?

**Question**:
Pick a city at random and tell me the weather there today.


**Assistant:**


**Search Result:**
- **Title:** [Weather in Fukuoka, Japan](https://www.weatherapi.com/)
- **Score:** `0.9870764`


**Search Result:**
- **Title:** [2025 - Fukuoka-shi, Fukuoka, Japan Monthly Weather | AccuWeather](https://www.accuweather.com/en/jp/fukuoka-shi/223544/august-weather/223544)
- **Score:** `0.98564`


**Assistant:**
I picked Fukuoka at random.

Current (local) conditions — 2025-08-26 00:14 JST (last updated 00:00):
- Condition: Partly cloudy  
- Temperature: 28.3°C (82.9°F) — feels like 34.8°C  
- Humidity: 79%  
- Wind: SSW 6.5 km/h (4.0 mph)  
- Pressure: 1012 mb  
- Precipitation: 0.0 mm, visibility 10 km

Source: WeatherAPI. Want the forecast for today or the weather for a different city?

## Replay full state history
We can see the full state history to understand everything that occurred.

In [10]:
for state in graph.get_state_history(config):
    print("Num messages: ", len(state.values["messages"]), "Next: ", state.next)
    print("-" * 80)

Num messages:  8 Next:  ()
--------------------------------------------------------------------------------
Num messages:  7 Next:  ('chatbot',)
--------------------------------------------------------------------------------
Num messages:  6 Next:  ('tools',)
--------------------------------------------------------------------------------
Num messages:  5 Next:  ('chatbot',)
--------------------------------------------------------------------------------
Num messages:  4 Next:  ('__start__',)
--------------------------------------------------------------------------------
Num messages:  4 Next:  ()
--------------------------------------------------------------------------------
Num messages:  3 Next:  ('chatbot',)
--------------------------------------------------------------------------------
Num messages:  2 Next:  ('__start__',)
--------------------------------------------------------------------------------
Num messages:  2 Next:  ()
-----------------------------------------------

## Resume from checkpoint
Langgraph saved checkpoints for every step of the graph. Let us resume the graph execution from step 4 and see the city that the LLM picks.

In [11]:
to_replay_idx = 4
for state in graph.get_state_history(config):
    if len(state.values["messages"]) == to_replay_idx:
        to_replay = state
        break

print(to_replay.next)
print(to_replay.config)

('__start__',)
{'configurable': {'thread_id': 'c7a93696-9278-4a0c-a6cd-d01b140fc6d7', 'checkpoint_ns': '', 'checkpoint_id': '1f081c64-efee-6dee-8005-1ad16d493e3c'}}


Let us now resume the graph from the checkpoint. 

In [12]:
stream_graph_updates(None, to_replay.config)

**Assistant:**


**Search Result:**
- **Title:** [Weather in Sapporo, Japan](https://www.weatherapi.com/)
- **Score:** `0.9973336`


**Search Result:**
- **Title:** [Sapporo-shi, Hokkaido, Japan Monthly Weather](https://www.accuweather.com/en/jp/sapporo-shi/223985/august-weather/223985)
- **Score:** `0.98553`


**Assistant:**
I picked Sapporo at random. Current conditions there (source: WeatherAPI, last updated 2025-08-26 00:15 JST):

- Condition: Partly cloudy  
- Temperature: 22.4 °C (72.3 °F), feels like 24.8 °C (76.6 °F)  
- Humidity: 88%  
- Wind: SSE at 15.5 km/h (9.6 mph), gusts to 26.9 km/h (16.7 mph)  
- Precipitation: 0.0 mm, visibility ~10 km, pressure 1014 mb

Would you like the forecast for the next few days or weather for a different city?

Previously, the LLM chose Fukuoka and this time, it opted for Sapporo.